<a href="https://colab.research.google.com/github/MettaTan/SIT-UofG-QC-Assignment/blob/main/BB84_Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=6aa27d7a96e4fe0a04eee90ac38b7a7b54a4e3f0a9d02fd39cd350cada50a05b
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.


## Overview

The BB84 protocol lets Alice and Bob share a secret key over a quantum channel.
Each step is implemented in one sequential program below, with sections labelled
**ALICE** and **BOB** to indicate which agent the code belongs to.

All random choices come from measuring qubits in the state
$|+\rangle = \tfrac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ —
no classical pseudo-random generator is used.


In [3]:
simulator = BasicSimulator()


## Quantum random bit generator

Measuring $|+\rangle$ in the computational basis returns `0` or `1` with equal probability.
We batch `n` such measurements into one circuit so we get `n` independent random bits per simulator call.


In [4]:
CHUNK = 16  # BasicSimulator's per-circuit qubit limit.

def quantum_random_bits(n):
    """Return a list of n uniformly random bits, each from measuring |+>.

    BasicSimulator caps at 24 qubits per circuit, so we batch in chunks.
    """
    bits = []
    while len(bits) < n:
        k = min(CHUNK, n - len(bits))
        qc = QuantumCircuit(k, k)
        qc.h(range(k))
        qc.measure(range(k), range(k))
        result = simulator.run(qc, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]
        # Qiskit's classical register is little-endian: qubit 0 is the rightmost character.
        bits.extend(int(b) for b in bitstring[::-1])
    return bits[:n]

# sanity check
print(quantum_random_bits(20))


[1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0]


## ALICE

Alice generates `N` random bits (her candidate key) and `N` random bases
(`0` = Z basis, `1` = X basis), then encodes each bit:

| basis | bit 0 | bit 1 |
|-------|-------|-------|
| Z (0) | $|0\rangle$ | $|1\rangle$ |
| X (1) | $|+\rangle$ | $|-\rangle$ |


In [5]:
N = 128  # number of qubits sent

# ----- ALICE -----
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

def prepare_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # |0> -> |1>
    if basis == 1:
        qc.h(0)          # rotate into the X basis
    return qc

alice_qubits = [prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]
print(f"Alice prepared {N} qubits.")


Alice prepared 128 qubits.


## BOB

Bob picks a random basis for each incoming qubit and measures.
For a Z measurement he measures directly; for X he applies $H$ first
(rotating the X basis back onto the Z basis before measuring).


In [6]:
# ----- BOB -----
bob_bases = quantum_random_bits(N)

def measure_qubit(qc, basis):
    """Measure a one-qubit circuit in basis 0 (Z) or 1 (X)."""
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])

bob_bits = [measure_qubit(alice_qubits[i], bob_bases[i]) for i in range(N)]
print(f"Bob measured {N} qubits.")


Bob measured 128 qubits.


## Public discussion — sifting

Alice and Bob announce their bases (but not their bits) over a public classical channel.
They keep only the positions where bases agreed; this is the **sifted key**.
About half the bits are discarded.


In [7]:
# ----- PUBLIC DISCUSSION -----
matching     = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
alice_sifted = [alice_bits[i] for i in matching]
bob_sifted   = [bob_bits[i]   for i in matching]

print(f"Sifted key length: {len(matching)} out of {N}")


Sifted key length: 65 out of 128


## Attack detection

Alice and Bob publicly reveal a subset of their sifted bits and count disagreements.
With no eavesdropper and no channel noise the error rate is 0.
An error rate above the threshold would indicate eavesdropping; the sample bits
are then discarded and the rest is the final key.


In [8]:
# Take the first half of the sifted bits as the sample.
# The selection is public so any fixed scheme is fine.
sample_size  = len(matching) // 2
sample_alice = alice_sifted[:sample_size]
sample_bob   = bob_sifted[:sample_size]

errors     = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate = errors / sample_size if sample_size else 0
THRESHOLD  = 0.15

print(f"Sample size:  {sample_size}")
print(f"Errors found: {errors}")
print(f"Error rate:   {error_rate:.2%}")
print(f"Threshold:    {THRESHOLD:.0%}")

if error_rate > THRESHOLD:
    print("\n>>> Eavesdropping suspected -- aborting.")
else:
    print("\n>>> Channel looks clean -- keeping the remaining sifted bits as the key.")


Sample size:  32
Errors found: 0
Error rate:   0.00%
Threshold:    15%

>>> Channel looks clean -- keeping the remaining sifted bits as the key.


## Final key


In [9]:
final_alice_key = alice_sifted[sample_size:]
final_bob_key   = bob_sifted[sample_size:]

print(f"Final key length: {len(final_alice_key)}")
print(f"Alice's key: {''.join(map(str, final_alice_key))}")
print(f"Bob's key:   {''.join(map(str, final_bob_key))}")
print(f"Keys match:  {final_alice_key == final_bob_key}")


Final key length: 33
Alice's key: 011010011100001001011101010111010
Bob's key:   011010011100001001011101010111010
Keys match:  True
